# 07 — MLP Classifier

Trains and evaluates a Multi-Layer Perceptron across the three feature spaces built in notebook 03.

The vault split is **locked** and not touched here — it is reserved for final evaluation in notebook 08.

## Setup

In [ ]:
import sys
sys.path.append('..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from config import TARGET_CLASSES, RANDOM_STATE

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
os.makedirs('../figures/07-MLP', exist_ok=True)

## Load Data

In [ ]:
X_train      = joblib.load('../data/processed/X_train.pkl')
y_train      = joblib.load('../data/processed/y_train.pkl')
feature_sets = joblib.load('../data/processed/feature_sets.pkl')

print(f"Train : {X_train.shape}")
print(f"\nClass distribution (train):")
vc = pd.Series(y_train.values).value_counts().sort_index()
for idx, cnt in vc.items():
    print(f"  {idx} ({TARGET_CLASSES[idx]:<25}) — {cnt} samples")
print(f"\nFeature sets loaded: {list(feature_sets.keys())}")

## Class Balance Strategy

The training classes are slightly imbalanced. Rather than resampling, we use
`compute_sample_weight('balanced')` **inside each CV fold** so the loss function
up-weights under-represented classes. `MLPClassifier` does not accept
`class_weight` directly, but it does accept `sample_weight` in `fit()`.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

preview_weights = compute_sample_weight('balanced', y_train)
print(f"Sample weight range: {preview_weights.min():.3f} → {preview_weights.max():.3f}")
print(f"(Higher = rarer class, lower = more common class)")

## Cross-Validation Setup

**Hierarchical CV scheme:**
- 20% of the total dataset is held out as the **vault** — never loaded here
- All model selection happens via **5-fold stratified CV** on the remaining 80%
- Transformers are **re-fit inside each fold** on that fold's training portion only — prevents data leakage
- Sample weights are recomputed fresh inside each fold

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def cv_evaluate(X_raw, y, model, transformer=None, use_sample_weight=True, verbose=False):
    """
    5-fold stratified CV.

    Parameters
    ----------
    X_raw       : raw feature array (pre-transformation)
    y           : labels
    model       : sklearn classifier
    transformer : optional sklearn transformer re-fit per fold to prevent leakage
    """
    y_arr = np.array(y)
    accs, f1s = [], []

    for fold, (tr_idx, val_idx) in enumerate(CV.split(X_raw, y_arr)):
        X_tr, X_val = X_raw[tr_idx], X_raw[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        if transformer is not None:
            t = clone(transformer)
            X_tr  = t.fit_transform(X_tr, y_tr)
            X_val = t.transform(X_val)

        fit_kwargs = {}
        if use_sample_weight:
            fit_kwargs['sample_weight'] = compute_sample_weight('balanced', y_tr)

        m = clone(model)
        m.fit(X_tr, y_tr, **fit_kwargs)
        y_pred = m.predict(X_val)

        accs.append(accuracy_score(y_val, y_pred))
        f1s.append(f1_score(y_val, y_pred, average='macro'))

        if verbose:
            print(f"  Fold {fold+1}: acc={accs[-1]:.4f}  macro-F1={f1s[-1]:.4f}")

    return {
        'acc_mean': np.mean(accs), 'acc_std': np.std(accs),
        'f1_mean' : np.mean(f1s),  'f1_std' : np.std(f1s),
    }

print("CV helper ready — StratifiedKFold(n_splits=5)")

## MLP — Multi-Layer Perceptron

Architecture: **128 → 64 → 7** (ReLU, Adam, early stopping).
Evaluated on each of the three feature spaces from notebook 03.

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    batch_size='auto',
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
)

## Run Cross-Validation

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

# Fresh transformer instances cloned and re-fit per fold — prevents leakage.
cv_transformers = {
    'original': None,
    'pca'     : PCA(n_components=0.95, random_state=RANDOM_STATE),
    'lasso'   : SelectFromModel(
                    LogisticRegression(solver='saga', l1_ratio=1, C=0.1,
                                       max_iter=2000, random_state=RANDOM_STATE,
                                       class_weight='balanced'),
                    threshold=1e-6
                ),
}

cv_results = {}

for fs_name, fs in feature_sets.items():
    print(f"\n── MLP on '{fs_name}' ({fs['n_features']} features) ──")
    scores = cv_evaluate(
        X_train.values, y_train, mlp,
        transformer=cv_transformers[fs_name],
        use_sample_weight=True, verbose=True
    )
    cv_results[fs_name] = scores
    print(f"  → Accuracy : {scores['acc_mean']:.4f} ± {scores['acc_std']:.4f}")
    print(f"  → Macro F1 : {scores['f1_mean']:.4f} ± {scores['f1_std']:.4f}")

## Results Summary

In [ ]:
summary_rows = []
for fs_name, scores in cv_results.items():
    summary_rows.append({
        'Feature Set' : fs_name,
        'N Features'  : feature_sets[fs_name]['n_features'],
        'CV Accuracy' : f"{scores['acc_mean']:.4f} ± {scores['acc_std']:.4f}",
        'CV Macro F1' : f"{scores['f1_mean']:.4f} ± {scores['f1_std']:.4f}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
fs_labels = list(cv_results.keys())
acc_means = [cv_results[k]['acc_mean'] for k in fs_labels]
acc_stds  = [cv_results[k]['acc_std']  for k in fs_labels]
f1_means  = [cv_results[k]['f1_mean']  for k in fs_labels]
f1_stds   = [cv_results[k]['f1_std']   for k in fs_labels]

x     = np.arange(len(fs_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, acc_means, width, yerr=acc_stds, capsize=4,
               label='Accuracy', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, f1_means,  width, yerr=f1_stds,  capsize=4,
               label='Macro F1', color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f.upper() for f in fs_labels])
ax.set_ylabel('Score')
ax.set_title('MLP — 5-Fold CV Performance by Feature Space')
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=8)
plt.tight_layout()
plt.savefig('../figures/07-MLP/mlp_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
joblib.dump(cv_results, '../data/processed/cv_results.pkl')

# Re-train MLP on full training set for each feature space (for use in notebook 08)
trained_models = {}
for fs_name, fs in feature_sets.items():
    m  = clone(mlp)
    sw = compute_sample_weight('balanced', y_train)
    m.fit(fs['X_train'], y_train.values, sample_weight=sw)
    trained_models[fs_name] = m
    print(f"Trained MLP ({fs_name}) — iterations: {m.n_iter_}")

joblib.dump(trained_models, '../data/processed/trained_models.pkl')
print("\nResults and models saved to data/processed/")